# Reproduce the scored solar filament submission

Install the dependencies described in the repository README. Extract the Kaggle
competition archive into `data/` and place the exported inference artifacts in
`models/`. This notebook uses the competition test images and saved models to
generate the predictions. It then assigns every shared pixel to the higher-ranked
mask and verifies the result against the submitted reference. No model is retrained.

The regenerated file is `outputs/reproduced/submission_no_overlap.csv`. If the
reference comparison fails, consult `models/runtime.txt` and use the original
software and GPU environment where possible.

In [ ]:
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
import gc
import hashlib
import json
import math
import sys
from time import perf_counter
from importlib.metadata import version

import cv2
from lightgbm import Booster
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from pycocotools import mask as coco
from sklearn.model_selection import GroupShuffleSplit
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection.roi_heads import paste_masks_in_image

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "remove_overlap.py").is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "remove_overlap.py").is_file():
    raise FileNotFoundError("Run from the repository root or its notebooks directory.")
sys.path.insert(0, str(REPO_ROOT))
from remove_overlap import repair_csv

INPUT_ROOT = REPO_ROOT / "data"  # Extract the competition data here or set an absolute path.
MODEL_DIR = REPO_ROOT / "models"
ARTIFACT_DIR = REPO_ROOT / "outputs" / "reproduced"
required_models = ("maskrcnn_weights.pt", "tiled_cnn_weights.pt", "lightgbm_model.txt", "inference_config.json")
missing_models = [name for name in required_models if not (MODEL_DIR / name).is_file()]
if missing_models:
    raise FileNotFoundError(f"Missing release assets in {MODEL_DIR}: {', '.join(missing_models)}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset folder missing: {INPUT_ROOT}. Set INPUT_ROOT to the extracted competition folder.")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is unavailable. Select the Jupyter kernel with CUDA-enabled PyTorch.")
torch.set_num_threads(4)
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = DEVICE.type != "cuda"
if DEVICE.type == "cuda":
    torch.use_deterministic_algorithms(False)
print(f"Python | {sys.executable}", flush=True)
gpu = torch.cuda.get_device_properties(DEVICE)
print(f"CUDA | {gpu.name} | {gpu.total_memory / 1024**3:.1f} GiB VRAM", flush=True)
print(f"Input directory: {INPUT_ROOT} | Output directory: {ARTIFACT_DIR}", flush=True)
print(f"Inference files: {MODEL_DIR}", flush=True)

## Load the competition test images

In [ ]:
def one_input(matches, label):
    paths = sorted(set(matches))
    if len(paths) != 1:
        raise FileNotFoundError(f"Expected one {label}, found {len(paths)}. Attach the competition dataset.")
    return paths[0]

test_image_dir = one_input(
    (p for p in INPUT_ROOT.rglob("test_images") if p.is_dir()), "test_images directory"
)

def read_image(path):
    with Image.open(path) as image:
        array = np.array(image.convert("L"),dtype=np.uint8)
    if array.shape != (2048,2048):
        raise ValueError(f"{path.name}: expected a native 2048 × 2048 Hα image")
    return array

## Define the detector and restore its configuration

In [ ]:
"""Original eight image-only neighbourhood channels; used by the Mask R-CNN adapter.
The following model core adds a ninth smooth illumination-corrected view.
Raw intensity is retained. These are evidence features, not calibrated probabilities.
No masks or instance IDs enter input feature generation.
"""
from dataclasses import asdict, dataclass
import math

import numpy as np
from scipy import ndimage as ndi


FEATURE_VERSION = "neighbour_v1"
FEATURE_NAMES = (
    "intensity",
    "local_darkness_fine",
    "local_darkness_medium",
    "local_darkness_broad",
    "dark_neighbour_support",
    "line_continuity",
    "bilateral_ridge",
    "contrast_edge",
)

# One score per source pixel and offset. Positive means the endpoint and source
# belong to the same foreground instance. These are learned outputs, not inputs.
AFFINITY_OFFSETS = ((0, 1), (1, 0), (1, 1), (1, -1))


@dataclass(frozen=True)
class NeighbourConfig:
    darkness_sigmas: tuple = (2.0, 4.0, 8.0)
    geometry_sigma: float = 0.8
    geometry_radii: tuple = (2, 4, 8)
    continuity_radii: tuple = (1, 2, 4)
    contrast_floor: float = 0.03
    gaussian_truncate: float = 3.0

    def __post_init__(self):
        if len(self.darkness_sigmas) != 3:
            raise ValueError("Exactly three darkness scales are required.")
        if any(not np.isfinite(s) or s <= 0 for s in self.darkness_sigmas):
            raise ValueError("Darkness scales must be finite and positive.")
        if tuple(sorted(self.darkness_sigmas)) != tuple(self.darkness_sigmas):
            raise ValueError("Darkness scales must be ordered fine to broad.")
        for radii in (self.geometry_radii, self.continuity_radii):
            if not radii or any(int(r) != r or r < 1 for r in radii):
                raise ValueError("Neighbour radii must be positive integers.")
        for value in (self.geometry_sigma, self.contrast_floor, self.gaussian_truncate):
            if not np.isfinite(value) or value <= 0:
                raise ValueError("Smoothing, contrast floor and truncation must be positive.")


DEFAULT_CONFIG = NeighbourConfig()


def feature_spec(config=DEFAULT_CONFIG):
    """Save this alongside a checkpoint and reuse it at inference."""
    return {
        "version": FEATURE_VERSION,
        "channels": list(FEATURE_NAMES),
        "config": asdict(config),
        "halo": feature_halo(config),
        "meaning": "image-derived soft evidence; not filament probabilities",
    }


def feature_halo(config=DEFAULT_CONFIG):
    """Finite support needed to reproduce full-image features on a tile."""
    # Match scipy.ndimage Gaussian's default radius rounding exactly.
    darkness = int(config.gaussian_truncate * max(config.darkness_sigmas) + 0.5)
    smooth = int(config.gaussian_truncate * config.geometry_sigma + 0.5)
    return max(
        darkness + max(config.continuity_radii),
        darkness + 1,  # 3x3 neighbour averaging of the darkness map.
        smooth + max(config.geometry_radii),
    )


def _normalise_image(image):
    value = np.asarray(image)
    if value.ndim != 2 or min(value.shape) < 1:
        raise ValueError("Expected a nonempty grayscale [H,W] image.")
    if value.dtype == np.uint8:
        value = value.astype(np.float64) / 255.0
    elif np.issubdtype(value.dtype, np.floating):
        value = value.astype(np.float64, copy=False)
    else:
        raise TypeError("Use uint8, or explicitly normalised floating-point pixels in [0,1].")
    if not np.isfinite(value).all() or value.min() < 0 or value.max() > 1:
        raise ValueError("Image pixels must be finite and in [0,1].")
    return value


def _score(positive_contrast):
    """Smooth monotonic compression; zero evidence stays zero."""
    positive = np.maximum(positive_contrast, 0.0)
    return positive / (1.0 + positive)


def _sample(image, dy, dx):
    """Read I[y+dy,x+dx], with reflected physical boundaries; never wrap."""
    return ndi.shift(image, shift=(-dy, -dx), order=0, mode="reflect", prefilter=False)


def _directions(radius):
    diagonal = max(1, int(round(radius / math.sqrt(2))))
    return ((0, radius), (radius, 0), (diagonal, diagonal), (diagonal, -diagonal))


def filament_neighbour_maps(image, config=DEFAULT_CONFIG):
    """Return eight [H,W] float32 maps using image pixels only.

    local_darkness: positive (local mean - intensity)/(local std + floor).
    dark_neighbour_support: dark evidence supported by the eight adjacent pixels.
    line_continuity: dark evidence extending on BOTH sides along some direction.
    bilateral_ridge: a dark center with brighter samples on BOTH sides across it.
    contrast_edge: intensity difference between opposite sides of a pixel.

    A strong contrast_edge can be a solar texture edge, limb, sunspot edge or
    filament edge. It is deliberately not called an instance-boundary probability.
    High line_continuity does not prove two regions share an annotated instance.
    """
    values = _normalise_image(image)
    square = values * values
    dark = []
    broad_std = None
    for sigma in config.darkness_sigmas:
        mean = ndi.gaussian_filter(values, sigma, mode="reflect", truncate=config.gaussian_truncate)
        second = ndi.gaussian_filter(square, sigma, mode="reflect", truncate=config.gaussian_truncate)
        std = np.sqrt(np.maximum(second - mean * mean, 0.0))
        # Avoid amplifying floating-point roundoff on constant images.
        difference = np.maximum(mean - values - 1e-12, 0.0)
        dark.append(_score(difference / (std + config.contrast_floor)))
        broad_std = std

    darkness = np.maximum.reduce(dark)
    neighbour_mean = (ndi.uniform_filter(darkness, size=3, mode="reflect") * 9 - darkness) / 8
    supported = np.sqrt(darkness * np.maximum(neighbour_mean, 0.0))

    # Integer reflected samples are views into one padded map, rather than
    # dozens of full-image interpolation calls. scipy reflect == numpy symmetric.
    def sampler(array, radius):
        padded = np.pad(array, ((radius, radius), (radius, radius)), mode="symmetric")
        height, width = array.shape
        return lambda dy, dx: padded[radius+dy:radius+dy+height, radius+dx:radius+dx+width]
    sample_dark = sampler(darkness, max(config.continuity_radii))
    continuity = np.zeros_like(values)
    for radius in config.continuity_radii:
        for dy, dx in _directions(radius):
            response = np.minimum(darkness, np.minimum(
                sample_dark(dy, dx), sample_dark(-dy, -dx)))
            np.maximum(continuity, response, out=continuity)

    smooth = ndi.gaussian_filter(values, config.geometry_sigma, mode="reflect", truncate=config.gaussian_truncate)
    sample_smooth = sampler(smooth, max(config.geometry_radii))
    denominator = broad_std + config.contrast_floor
    ridge = np.zeros_like(values)
    edge = np.zeros_like(values)
    for radius in config.geometry_radii:
        for dy, dx in _directions(radius):
            side_a = sample_smooth(dy, dx)
            side_b = sample_smooth(-dy, -dx)
            ridge_response = _score(np.minimum(side_a - smooth, side_b - smooth) / denominator)
            edge_response = _score(np.abs(side_a - side_b) / denominator)
            np.maximum(ridge, ridge_response, out=ridge)
            np.maximum(edge, edge_response, out=edge)

    arrays = (values, *dark, supported, continuity, ridge, edge)
    return {
        name: np.ascontiguousarray(np.clip(array, 0, 1), dtype=np.float32)
        for name, array in zip(FEATURE_NAMES, arrays)
    }


def filament_features(image, config=DEFAULT_CONFIG):
    """Return float32 [8,H,W] in the fixed FEATURE_NAMES order."""
    maps = filament_neighbour_maps(image, config)
    return np.stack([maps[name] for name in FEATURE_NAMES])


def filament_feature_crop(image, y, x, size=512, config=DEFAULT_CONFIG):
    """Compute features with REAL halo context, then return a central crop.

    Origins must lie inside the source image. Crops extending past bottom/right
    are edge-padded after feature extraction; exclude these padded pixels from
    all supervised losses. At actual image edges, reflection matches the full
    image routine. This helper performs no data augmentation.
    """
    values = _normalise_image(image)
    height, width = values.shape
    if any(int(v) != v for v in (y, x, size)):
        raise ValueError("Crop coordinates and size must be integers.")
    y, x, size = int(y), int(x), int(size)
    if not (0 <= y < height and 0 <= x < width and size > 0):
        raise ValueError("Invalid crop origin or size.")
    halo = feature_halo(config)
    top, left = max(0, y - halo), max(0, x - halo)
    bottom, right = min(height, y + size + halo), min(width, x + size + halo)
    expanded = filament_features(values[top:bottom, left:right], config)
    result = expanded[:, y-top:y-top+size, x-left:x-left+size]
    return np.ascontiguousarray(np.pad(
        result, ((0, 0), (0, size-result.shape[1]), (0, size-result.shape[2])), mode="edge"))



"""Native-resolution solar Mask R-CNN, residual FPN attention and mask IoU loss."""
import copy
import math
import json
from collections import OrderedDict
from pathlib import Path
import numpy as np
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.rpn import RPNHead
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.roi_heads import RoIHeads, fastrcnn_loss, maskrcnn_inference, project_masks_on_boxes
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.ops import MultiScaleRoIAlign, box_iou
from torchvision.ops.misc import FrozenBatchNorm2d
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from pycocotools import mask as coco_mask

MODEL_FEATURE_VERSION = "mask_neighbour_illum_v1"
MODEL_FEATURE_NAMES = tuple(FEATURE_NAMES) + ("illumination_corrected",)


def model_feature_spec():
    return dict(version=MODEL_FEATURE_VERSION, channels=list(MODEL_FEATURE_NAMES),
                neighbourhood=feature_spec(), illumination_sigma_native=64.0,
                note="Image-derived evidence, not probabilities or additional physical measurements.")


def illumination_corrected(image):
    """Smooth image-only illumination estimate; raw intensity remains available.

    This is NOT a calibrated physical limb-darkening inversion. The low-resolution
    normalised convolution avoids zero-space depressing the background estimate.
    """
    values = _normalise_image(image).astype(np.float32)
    stride = 8
    small = values[::stride, ::stride]
    cutoff = max(0.03, 0.25 * float(np.quantile(small, .95)))
    labels, count = ndi.label(small > cutoff)
    if count == 0:
        return values.copy()
    areas = np.bincount(labels.ravel()); areas[0] = 0
    disk = ndi.binary_fill_holes(labels == int(areas.argmax()))
    weight = ndi.gaussian_filter(disk.astype(np.float32), 64.0 / stride, mode="reflect")
    numerator = ndi.gaussian_filter(small * disk, 64.0 / stride, mode="reflect")
    background = numerator / np.maximum(weight, 1e-4)
    # Coordinate interpolation is deterministic for all native dimensions.
    y = np.arange(values.shape[0], dtype=np.float32) / stride
    x = np.arange(values.shape[1], dtype=np.float32) / stride
    yy, xx = np.meshgrid(y, x, indexing="ij")
    expanded = ndi.map_coordinates(background, [yy, xx], order=1, mode="nearest")
    support = ndi.map_coordinates(disk.astype(np.float32), [yy, xx], order=0, mode="nearest") > .5
    corrected = np.clip(.5 + .5 * (values / np.maximum(expanded, .1) - 1.0), 0, 1)
    return np.ascontiguousarray(np.where(support, corrected, values), np.float32)


def model_features(image):
    original = filament_features(image)
    corrected = illumination_corrected(image)
    result = np.concatenate([original, corrected[None]], axis=0)
    assert result.shape[0] == len(MODEL_FEATURE_NAMES) and np.isfinite(result).all()
    return np.ascontiguousarray(result, np.float32)


class ResidualCBAM(nn.Module):
    """Channel/spatial attention with bounded residual modulation.

    The identity path cannot be removed by an attention gate. Initially attention
    only mildly modulates the pretrained FPN, and every attention branch is live.
    """
    def __init__(self, channels=256, strength=.25):
        super().__init__()
        hidden = max(8, channels // 16)
        self.channel = nn.Sequential(nn.Conv2d(channels, hidden, 1), nn.ReLU(), nn.Conv2d(hidden, channels, 1))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3)
        self.scale = nn.Parameter(torch.tensor(.1))
        self.strength = float(strength)

    def forward(self, x):
        channel = torch.sigmoid(self.channel(F.adaptive_avg_pool2d(x, 1)) +
                                self.channel(F.adaptive_max_pool2d(x, 1)))
        refined = x * channel
        spatial = torch.sigmoid(self.spatial(torch.cat([refined.mean(1, keepdim=True),
                                                       refined.amax(1, keepdim=True)], dim=1)))
        attention = channel * spatial
        return x * (1 + self.strength * torch.tanh(self.scale) * (2 * attention - 1))


class SolarBackbone(nn.Module):
    def __init__(self, backbone, config):
        super().__init__()
        self.input_adapter = nn.Conv2d(len(MODEL_FEATURE_NAMES), 3, 1, bias=False)
        with torch.no_grad():
            self.input_adapter.weight.fill_(.005)
            self.input_adapter.weight[:, 0, 0, 0] = 1.0
        self.body, self.fpn = backbone.body, backbone.fpn
        self.checkpoint_body = bool(config.get("backbone_checkpointing", True))
        self.out_channels = backbone.out_channels
        self.attention = nn.ModuleDict({k: ResidualCBAM(self.out_channels, config["attention_strength"])
                                        for k in ["0", "1", "2", "3", "pool"]})
        self.register_buffer("rgb_mean", torch.tensor([.485, .456, .406])[None, :, None, None])
        self.register_buffer("rgb_std", torch.tensor([.229, .224, .225])[None, :, None, None])

    def forward(self, x):
        rgb = self.input_adapter(x).clamp(0, 1)
        value = (rgb - self.rgb_mean) / self.rgb_std
        if self.training and self.checkpoint_body and any(p.requires_grad for p in self.body.parameters()):
            outputs = OrderedDict()
            for name, layer in self.body.items():
                value = activation_checkpoint(layer, value, use_reentrant=False) if name.startswith("layer") else layer(value)
                if name in self.body.return_layers:
                    outputs[self.body.return_layers[name]] = value
        else:
            outputs = self.body(value)
        features = self.fpn(outputs)
        return OrderedDict((key, self.attention[key](value)) for key, value in features.items())


class NativeTransform(GeneralizedRCNNTransform):
    def __init__(self):
        super().__init__(min_size=2048, max_size=2048, image_mean=[0.] * len(MODEL_FEATURE_NAMES),
                         image_std=[1.] * len(MODEL_FEATURE_NAMES), size_divisible=32)

    def resize(self, image, target=None):
        # No hidden 2048 -> 800 shrink, and no resized training targets.
        return image, target

    def postprocess(self, result, image_shapes, original_image_sizes):
        if self.training:
            return result
        # Paste ROI masks on CPU to avoid large N x 2048 x 2048 GPU allocations.
        cpu_result = [{k: v.detach().float().cpu() if k in ["boxes", "scores", "masks"] else v.detach().cpu()
                       for k, v in item.items()} for item in result]
        return super().postprocess(cpu_result, image_shapes, original_image_sizes)


def lovasz_hinge_per_roi(logits, target):
    if logits.numel() == 0:
        return logits.sum() * 0
    logits = logits.float().flatten(1)
    truth = (target >= .5).float().flatten(1)
    errors = 1 - logits * (2 * truth - 1)
    errors, permutation = torch.sort(errors, dim=1, descending=True)
    sorted_truth = truth.gather(1, permutation)
    positive = sorted_truth.sum(1, keepdim=True)
    intersection = positive - sorted_truth.cumsum(1)
    union = positive + (1 - sorted_truth).cumsum(1)
    jaccard = 1 - intersection / union.clamp_min(1)
    gradient = torch.cat([jaccard[:, :1], jaccard[:, 1:] - jaccard[:, :-1]], dim=1)
    return (F.relu(errors) * gradient).sum(1).mean()


class SolarRoIHeads(RoIHeads):
    def __init__(self, old, lovasz_weight):
        super().__init__(old.box_roi_pool, old.box_head, old.box_predictor,
                         old.proposal_matcher.high_threshold, old.proposal_matcher.low_threshold,
                         old.fg_bg_sampler.batch_size_per_image, old.fg_bg_sampler.positive_fraction,
                         old.box_coder.weights, old.score_thresh, old.nms_thresh, old.detections_per_img,
                         mask_roi_pool=old.mask_roi_pool, mask_head=old.mask_head, mask_predictor=old.mask_predictor)
        self.lovasz_weight = float(lovasz_weight)
        self.last_mask_components = {}

    def forward(self, features, proposals, image_shapes, targets=None):
        if self.training:
            proposals, matched, labels, regression = self.select_training_samples(proposals, targets)
        else:
            matched = labels = regression = None
        box_features = self.box_head(self.box_roi_pool(features, proposals, image_shapes))
        class_logits, box_regression = self.box_predictor(box_features)
        result, losses = [], {}
        if self.training:
            cls, box = fastrcnn_loss(class_logits, box_regression, labels, regression)
            losses.update(loss_classifier=cls, loss_box_reg=box)
            positive = [torch.where(x > 0)[0] for x in labels]
            mask_proposals = [p[i] for p, i in zip(proposals, positive)]
            mask_matched = [m[i] for m, i in zip(matched, positive)]
        else:
            boxes, scores, predicted_labels = self.postprocess_detections(class_logits, box_regression, proposals, image_shapes)
            result = [dict(boxes=b, scores=s, labels=l) for b, s, l in zip(boxes, scores, predicted_labels)]
            mask_proposals = [x["boxes"] for x in result]
        mask_features = self.mask_head(self.mask_roi_pool(features, mask_proposals, image_shapes))
        logits = self.mask_predictor(mask_features)
        if self.training:
            mask_labels = torch.cat([t["labels"][idx] for t, idx in zip(targets, mask_matched)])
            mask_targets = torch.cat([project_masks_on_boxes(t["masks"], boxes, idx, logits.shape[-1])
                                      for t, boxes, idx in zip(targets, mask_proposals, mask_matched)])
            if mask_targets.numel():
                selected = logits[torch.arange(len(mask_labels), device=logits.device), mask_labels].float()
                bce = F.binary_cross_entropy_with_logits(selected, mask_targets.float())
                lovasz = lovasz_hinge_per_roi(selected, mask_targets.float())
                target_empty = (mask_targets.flatten(1).sum(1) == 0).float().mean()
            else:
                bce = lovasz = logits.sum() * 0
                target_empty = logits.new_zeros(())
            losses["loss_mask"] = bce + self.lovasz_weight * lovasz
            self.last_mask_components = dict(mask_bce=bce.detach(), mask_lovasz=lovasz.detach(),
                                             empty_mask_target_fraction=target_empty.detach())
        else:
            probabilities = maskrcnn_inference(logits, [r["labels"] for r in result])
            for row, masks in zip(result, probabilities):
                row["masks"] = masks
        return result, losses


def convert_frozen_bn(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            with torch.no_grad():
                for key in ["weight", "bias", "running_mean", "running_var"]:
                    getattr(frozen, key).copy_(getattr(child, key))
            setattr(module, name, frozen)
        else:
            convert_frozen_bn(child)


def record_proposals(module, inputs, output):
    if not module.training:
        module.last_proposals = [p.detach().float().cpu() for p in output[0]]


def build_solar_maskrcnn(config, initialize=False, pretrained_path=None):
    """Construction never downloads weights during checkpoint reload/inference."""
    model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
    convert_frozen_bn(model)
    if initialize and config["initialization"] == "coco":
        if pretrained_path is None:
            raise ValueError("COCO initialization requires the local COCO model weights.")
        state = torch.load(pretrained_path,map_location="cpu",weights_only=True)
        model.load_state_dict(state,strict=True)
    elif initialize and config["initialization"] != "random":
        raise ValueError("initialization must be coco or explicitly random")
    original_rpn = model.rpn.head
    ratios = tuple(float(v) for v in config["anchor_ratios"])
    model.rpn.anchor_generator = AnchorGenerator(tuple((float(s),) for s in config["anchor_sizes"]),
                                                (ratios,) * 5)
    replacement = RPNHead(256, len(ratios))
    replacement.conv.load_state_dict(original_rpn.conv.state_dict())
    with torch.no_grad():
        for j, ratio in enumerate(ratios):
            k = int(np.argmin(np.abs(np.log(np.array([.5, 1., 2.])) - np.log(ratio))))
            replacement.cls_logits.weight[j].copy_(original_rpn.cls_logits.weight[k])
            replacement.cls_logits.bias[j].copy_(original_rpn.cls_logits.bias[k])
            replacement.bbox_pred.weight[4*j:4*j+4].copy_(original_rpn.bbox_pred.weight[4*k:4*k+4])
            replacement.bbox_pred.bias[4*j:4*j+4].copy_(original_rpn.bbox_pred.bias[4*k:4*k+4])
    model.rpn.head = replacement
    model.rpn._pre_nms_top_n = dict(training=config["rpn_pre_train"], testing=config["rpn_pre_test"])
    model.rpn._post_nms_top_n = dict(training=config["rpn_post_train"], testing=config["rpn_post_test"])
    model.roi_heads.fg_bg_sampler.batch_size_per_image = config["roi_batch_size"]
    model.roi_heads.fg_bg_sampler.positive_fraction = .25
    model.roi_heads.score_thresh = config["proposal_score_floor"]
    model.roi_heads.nms_thresh = config["box_nms_threshold"]
    model.roi_heads.detections_per_img = config["detections_per_image"]
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 2)
    old_predictor = model.roi_heads.mask_predictor
    model.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, 2)
    model.roi_heads.mask_predictor.conv5_mask.load_state_dict(old_predictor.conv5_mask.state_dict())
    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(["0", "1", "2", "3"], config["mask_pool_size"], 2)
    model.roi_heads = SolarRoIHeads(model.roi_heads, config["mask_lovasz_weight"])
    model.backbone = SolarBackbone(model.backbone, config)
    model.transform = NativeTransform()
    model.rpn.register_forward_hook(record_proposals)
    return model


@torch.inference_mode()
def predict_native_mask(model, image, config, *, prepared_features=None,
                        preprocess_seconds=None, score_floor=None):
    model.eval()
    device = next(model.parameters()).device
    started = __import__("time").perf_counter()
    features = model_features(image) if prepared_features is None else prepared_features
    prep_seconds = (__import__("time").perf_counter() - started
                    if preprocess_seconds is None else float(preprocess_seconds))
    tensor = (torch.from_numpy(features) if isinstance(features, np.ndarray) else features).to(device, non_blocking=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
    started = __import__("time").perf_counter()
    original_floor = model.roi_heads.score_thresh
    # torchvision uses strict > while our final postprocessing retains >=.
    # One float32 predecessor retains equality at the final confidence cutoff.
    if score_floor is not None:
        if float(score_floor) < original_floor:
            raise ValueError("Requested confidence is below the model's proposal floor")
        model.roi_heads.score_thresh = max(original_floor, float(np.nextafter(
            np.float32(score_floor), np.float32(-np.inf))))
    try:
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=bool(config["inference_amp"] and device.type == "cuda")):
            result = model([tensor])[0]
    finally:
        model.roi_heads.score_thresh = original_floor
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = __import__("time").perf_counter() - started
    masks = result["masks"][:, 0].float().numpy()
    scores = result["scores"].float().numpy()
    boxes = result["boxes"].float().numpy()
    if not np.isfinite(masks).all() or not np.isfinite(scores).all():
        raise FloatingPointError("Nonfinite native prediction")
    return dict(masks=masks, scores=scores, boxes=boxes,
                proposals=model.rpn.last_proposals[0].numpy(),
                preprocess_seconds=prep_seconds, inference_and_paste_seconds=elapsed)

In [ ]:
metadata = json.loads((MODEL_DIR / "inference_config.json").read_text())
CFG = metadata["detector_config"]

def keep_roi_probabilities(result, image_shapes, original_sizes):
    return [{key: value.detach().float().cpu() if key in {"boxes", "scores", "masks"}
             else value.detach().cpu() for key, value in item.items()} for item in result]

## Define the tiled CNN and tile blending

In [ ]:
TILE = 768
TILE_SIDE = int(metadata["proposal_tile_side"])
PROPOSAL_INFERENCE_BATCH = int(metadata["proposal_inference_batch"])
POSITIONS = [(y,x) for y in (0,640,1280) for x in (0,640,1280)]
WINDOW = np.maximum(np.outer(np.hanning(TILE),np.hanning(TILE)),.05).astype(np.float32)
WEIGHT_SUM = np.zeros((2048,2048),dtype=np.float32)
for y,x in POSITIONS:
    WEIGHT_SUM[y:y+TILE,x:x+TILE] += WINDOW


def tiled_image(image):
    return np.stack([cv2.resize(image[y:y+TILE,x:x+TILE],(TILE_SIDE,TILE_SIDE),interpolation=cv2.INTER_AREA)
                     for y,x in POSITIONS])


def block(ch_in,ch_out):
    return nn.Sequential(nn.Conv2d(ch_in,ch_out,3,padding=1,bias=False),
                         nn.GroupNorm(4,ch_out),nn.SiLU(),
                         nn.Conv2d(ch_out,ch_out,3,padding=1,bias=False),
                         nn.GroupNorm(4,ch_out),nn.SiLU())


class TileProposalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = block(3,12)
        self.enc2 = block(12,24)
        self.center = block(24,32)
        self.dec2 = block(56,24)
        self.dec1 = block(36,12)
        self.head = nn.Conv2d(12,1,1)
        nn.init.constant_(self.head.bias,-2.)

    def forward(self,image):
        image = image[:,None].float()/255.
        fine = F.avg_pool2d(image,7,stride=1,padding=3,count_include_pad=False)
        broad = F.avg_pool2d(image,21,stride=1,padding=10,count_include_pad=False)
        e1 = self.enc1(torch.cat([image,10*(fine-image),5*(broad-image)],dim=1))
        e2 = self.enc2(F.max_pool2d(e1,2))
        core = self.center(F.max_pool2d(e2,2))
        d2 = self.dec2(torch.cat([e2,F.interpolate(core,size=e2.shape[-2:],mode="bilinear",align_corners=False)],1))
        d1 = self.dec1(torch.cat([e1,F.interpolate(d2,size=e1.shape[-2:],mode="bilinear",align_corners=False)],1))
        return self.head(d1)[:,0]


@torch.inference_mode()
def predict_probability(image):
    tiles = torch.from_numpy(tiled_image(image))
    probabilities = np.concatenate([
        proposal_model(batch.to(DEVICE)).sigmoid().cpu().numpy()
        for batch in tiles.split(PROPOSAL_INFERENCE_BATCH)
    ])
    result = np.zeros(image.shape,dtype=np.float32)
    for prob,(y,x) in zip(probabilities,POSITIONS,strict=True):
        result[y:y+TILE,x:x+TILE] += cv2.resize(prob,(TILE,TILE),interpolation=cv2.INTER_LINEAR)*WINDOW
    return (result/WEIGHT_SUM).astype(np.float16).astype(np.float32)

## Define candidate generation and fusion

In [ ]:
def rcnn_candidates(image, model=None, *, confidence=.40, mask_threshold=.40, minimum_area=512):
    model = mask_model if model is None else model
    raw = predict_native_mask(model, image, CFG, score_floor=confidence)
    accepted = []
    for i in np.argsort(-raw["scores"], kind="stable"):
        if raw["scores"][i] < confidence:
            continue
        roi = torch.from_numpy(raw["masks"][i])[None, None]
        box = torch.from_numpy(raw["boxes"][i])[None]
        mask = paste_masks_in_image(roi, box, image.shape)[0, 0].numpy() >= mask_threshold
        if np.count_nonzero(mask) < minimum_area:
            continue
        rle = coco.encode(np.asfortranarray(mask, dtype=np.uint8))
        if accepted and (coco.iou([rle], [c["rle"] for c in accepted], [0]*len(accepted))[0] > 0.70).any():
            continue
        accepted.append(dict(rle=rle, score=float(raw["scores"][i]), source=0))
    return accepted


def proposal_masks(prob):
    bw = (prob >= .40).astype('uint8')
    bw = cv2.morphologyEx(bw,cv2.MORPH_CLOSE,np.ones((3,3),'uint8'))
    n,labels,stats,_ = cv2.connectedComponentsWithStats(bw,connectivity=8)
    out = []
    canvas = np.zeros(prob.shape,dtype=np.uint8,order='F')
    for k in range(1,n):
        x,y,w,h,area = map(int,stats[k])
        if area < 256:
            continue
        region = np.s_[y:y+h,x:x+w]
        mask = labels[region]==k
        canvas[region] = mask
        out.append(dict(rle=coco.encode(canvas),score=float(np.quantile(prob[region][mask],.90)),source=1))
        canvas[region] = 0
    return sorted(out,key=lambda p:-p['score'])


def get_features(item,raw,prob,cross_iou):
    rle = item['rle']
    x,y,w,h = map(int,coco.toBbox(rle))
    y0,x0 = max(0,y-12),max(0,x-12)
    region = np.s_[y0:min(y+h+12,2048),x0:min(x+w+12,2048)]
    mask = coco.decode(rle)[region].astype(bool)
    area = int(mask.sum())
    pixels, pmap = raw[region],prob[region]
    ring = cv2.dilate(mask.astype('uint8'),np.ones((11,11),'uint8')).astype(bool)&~mask
    inside = pixels[mask].astype('float32')
    exterior = pixels[ring]
    local_contrast = (exterior.mean()-inside.mean())/(exterior.std()+4) if exterior.size else 0.
    values = pmap[mask]
    perimeter = cv2.findContours(mask.astype('uint8'),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]
    perimeter = sum(cv2.arcLength(c,True) for c in perimeter)
    return [item['source'],item['score'],np.log1p(area),np.log((w+1)/(h+1)),area/max(w*h,1),
            perimeter/np.sqrt(area),float(inside.mean())/255,float(inside.std())/255,local_contrast,
            float(values.mean()),float(np.quantile(values,.20)),float(np.quantile(values,.90)),
            float((values>=.40).mean()),cross_iou]


def merge(row, ordered, accepted=None):
    keep = list(accepted or [])
    for i in ordered:
        if not keep or not row['duplicate'][i,keep].any():
            keep.append(i)
    return keep


def duplicate_matrix(rles):
    pair_iou = coco.iou(rles, rles, [0] * len(rles))
    areas = np.array([coco.area(rle) for rle in rles], dtype=float)
    overlap = pair_iou * (areas[:, None] + areas[None, :]) / (1 + pair_iou)
    duplicate = overlap >= .50 * np.minimum(areas[:, None], areas[None, :])
    return duplicate


def fused_indices(scores, duplicate, cutoff):
    order = np.argsort(-scores, kind="stable")
    eligible = order[scores[order] >= cutoff]
    return merge({"duplicate": duplicate}, eligible)


def candidate_features(image, probability, base, proposed):
    candidates = base + proposed
    cross = np.zeros(len(candidates))
    if base and proposed:
        matrix = coco.iou([c["rle"] for c in base],[c["rle"] for c in proposed],[0]*len(proposed))
        cross = np.r_[matrix.max(1),matrix.max(0)]
    return np.asarray([get_features(c,image,probability,overlap)
                       for c,overlap in zip(candidates,cross,strict=True)]).reshape(-1,14)

## Load the exported model weights and cutoff

In [ ]:
CROP_SIDE = int(metadata["crop_side"])
CROP_PADDING = float(metadata["crop_padding"])
CROP_BATCH = int(metadata["crop_batch"])


class CandidateCropCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 12, 5, stride=2, padding=2), nn.SiLU(),
            nn.Conv2d(12, 24, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(24, 32, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(32, 48, 3, stride=2, padding=1), nn.SiLU(),
            nn.AdaptiveAvgPool2d((2, 2)), nn.Flatten(), nn.Linear(192, 1),
        )

    def forward(self, x):
        return self.network(x.float().div(255)).squeeze(1)


def candidate_crop(image, probability, rle):
    x, y, w, h = map(int, coco.toBbox(rle))
    pad_x, pad_y = max(12, math.ceil(w * CROP_PADDING)), max(12, math.ceil(h * CROP_PADDING))
    height, width = image.shape
    region = np.s_[max(0, y-pad_y):min(height, y+h+pad_y),
                   max(0, x-pad_x):min(width, x+w+pad_x)]
    patch = image[region]
    mask = coco.decode(rle)[region]
    pmap = np.rint(np.clip(probability[region], 0., 1.) * 255).astype(np.uint8)
    ph, pw = patch.shape
    scale = CROP_SIDE / max(ph, pw)
    rh, rw = max(1, round(ph * scale)), max(1, round(pw * scale))
    top, left = (CROP_SIDE-rh)//2, (CROP_SIDE-rw)//2
    result = np.zeros((3, CROP_SIDE, CROP_SIDE), dtype=np.uint8)
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    for channel, pixels, method in [(0, patch, interpolation), (1, mask * 255, cv2.INTER_NEAREST),
                                    (2, pmap, interpolation)]:
        result[channel, top:top+rh, left:left+rw] = cv2.resize(pixels, (rw, rh), interpolation=method)
    return result


@torch.inference_mode()
def predict_crop_quality(models, crops):
    if not len(crops):
        return np.empty(0, dtype=np.float32)
    values = []
    for batch in torch.from_numpy(crops).split(CROP_BATCH):
        batch = batch.to(DEVICE)
        values.append(torch.stack([model(batch).sigmoid() for model in models]).mean(0).cpu().numpy())
    return np.concatenate(values)


def image_variants(image):
    base = rcnn_candidates(image)
    probability = predict_probability(image)
    proposals = proposal_masks(probability)
    candidates = base + proposals
    rles = [item["rle"] for item in candidates]
    if not rles:
        return {name: [] for name in VARIANT_FILES}

    features = candidate_features(image, probability, base, proposals)
    crops = np.stack([candidate_crop(image, probability, rle) for rle in rles])
    quality = predict_crop_quality(crop_models, crops)
    duplicate = duplicate_matrix(rles)
    plain_scores = plain_booster.predict(features, num_threads=4)
    full_scores = verifier_booster.predict(np.column_stack([features, quality]), num_threads=4)
    selections = {
        "maskrcnn": range(len(base)),
        "tiled_only": range(len(base), len(rles)),
        "after_tiled": merge({"duplicate": duplicate}, range(len(rles))),
        "after_crop": merge({"duplicate": duplicate}, np.argsort(-quality, kind="stable")),
        "lightgbm_without_crop": fused_indices(plain_scores, duplicate, PLAIN_CUTOFF),
        "after_lightgbm": fused_indices(full_scores, duplicate, VERIFIER_CUTOFF),
    }
    return {name: [rles[i] for i in indices] for name, indices in selections.items()}

In [ ]:
mask_model = build_solar_maskrcnn(CFG, initialize=False, pretrained_path=None)
mask_model.load_state_dict(torch.load(MODEL_DIR / "maskrcnn_weights.pt",
                                     map_location="cpu", weights_only=True), strict=True)
mask_model = mask_model.to(DEVICE).eval()
mask_model.transform.postprocess = keep_roi_probabilities

proposal_model = TileProposalNet().to(DEVICE, memory_format=torch.channels_last)
proposal_model.load_state_dict(torch.load(MODEL_DIR / "tiled_cnn_weights.pt",
                                         map_location="cpu", weights_only=True), strict=True)
proposal_model.eval()
verifier_booster = Booster(model_file=str(MODEL_DIR / "lightgbm_model.txt"))
VERIFIER_CUTOFF = metadata["verifier_cutoff"]

crop_models = []
for name in metadata["crop_weight_files"]:
    model = CandidateCropCNN().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_DIR / name, map_location="cpu", weights_only=True), strict=True)
    crop_models.append(model.eval())
plain_booster = Booster(model_file=str(MODEL_DIR / "lightgbm_without_crop.txt"))
PLAIN_CUTOFF = float(metadata["plain_cutoff"])

## Regenerate and verify the CSV

In [ ]:
VARIANT_FILES = {
    "maskrcnn": "submission_maskrcnn.csv",
    "tiled_only": "submission_tiled_only.csv",
    "after_tiled": "submission_after_tiled.csv",
    "after_crop": "submission_after_crop.csv",
    "lightgbm_without_crop": "submission_lightgbm_without_crop.csv",
    "after_lightgbm": "submission.csv",
}


def export_submission():
    paths = sorted(p for p in test_image_dir.rglob("*")
                   if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"})
    if not paths or len({p.stem for p in paths}) != len(paths) or any("_" in p.stem for p in paths):
        raise ValueError("Expected unique competition test images")
    rows = {key: [] for key in VARIANT_FILES}
    audit = []
    for index, path in enumerate(paths, 1):
        masks_by_variant = image_variants(read_image(path))
        audit.append(dict(image=path.name, **{k: len(v) for k, v in masks_by_variant.items()}))
        for key, masks in masks_by_variant.items():
            for j, mask in enumerate(masks, 1):
                counts = mask["counts"]
                rows[key].append((f"{path.stem}_{j}", counts.decode("ascii") if isinstance(counts, bytes) else counts))
        if index == 1 or index % 10 == 0 or index == len(paths):
            print(f"Inferred {index}/{len(paths)} official test images", flush=True)
    for key, filename in VARIANT_FILES.items():
        table = pd.DataFrame(rows[key], columns=["filament_id", "segmentation_rle"])
        if not table.filament_id.is_unique or (key == "after_lightgbm" and table.empty):
            raise ValueError(f"Invalid competition predictions for {key}")
        path = ARTIFACT_DIR / filename
        table.to_csv(path, index=False)
        if not pd.read_csv(path, dtype=str, keep_default_na=False).equals(table):
            raise ValueError(f"CSV round trip failed for {filename}")
        print(f"{key} | {len(table)} predicted masks | {path}", flush=True)
    pd.DataFrame(audit).to_csv(ARTIFACT_DIR / "submission_manifest.csv", index=False)


export_submission()

raw_csv = ARTIFACT_DIR / "submission.csv"
corrected_csv = ARTIFACT_DIR / "submission_no_overlap.csv"
print(repair_csv(raw_csv, corrected_csv))

reference = REPO_ROOT / "results" / "submission.csv"
if corrected_csv.read_bytes() != reference.read_bytes():
    raise RuntimeError(
        "The regenerated CSV differs from the scored reference. "
        "Check the model files and the original CUDA environment."
    )
print("CSV VERIFIED | Reproduced the scored submission byte for byte", flush=True)